## Example 1: nkeys module structure

In [ ]:
# Example 1: nkeys module structure
import nkeys

print("Example 1: nkeys module structure")

# Show available components
components = [x for x in dir(nkeys) if not x.startswith('_')]
print(f"\nAvailable components ({len(components)}): {components[:10]}...")

# Show prefix bytes for NATS entities
print("\nNATS entity prefix bytes:")
print(f"  User: {nkeys.PREFIX_BYTE_USER}")
print(f"  Account: {nkeys.PREFIX_BYTE_ACCOUNT}")
print(f"  Server: {nkeys.PREFIX_BYTE_SERVER}")

# Show key functions
print("\nKey functions:")
print(f"  from_seed: {nkeys.from_seed}")
print(f"  KeyPair class: {nkeys.KeyPair}")

print("\nExample 1 completed successfully")

## Example 2: Prefix byte validation

In [ ]:
# Example 2: Prefix byte validation
import nkeys

print("Example 2: Prefix byte validation")

# Test prefix byte validation
print("Validating prefix bytes:")
test_bytes = [0, 16, 104, 112, 120, 144, 160, 200]

for byte_val in test_bytes:
    is_valid = nkeys.valid_prefix_byte(byte_val)
    is_public_valid = nkeys.valid_public_prefix_byte(byte_val)
    print(f"  Byte {byte_val:3d} (0x{byte_val:02x}): prefix_valid={is_valid}, public_valid={is_public_valid}")

# Show which ones are valid
valid_bytes = [b for b in range(256) if nkeys.valid_prefix_byte(b)]
print(f"\nTotal valid prefix bytes: {len(valid_bytes)}")
print(f"Valid bytes: {valid_bytes}")

valid_public = [b for b in range(256) if nkeys.valid_public_prefix_byte(b)]
print(f"Valid public prefix bytes: {valid_public}")

# Public keys may carry the user/account/server prefixes; seeds and private keys may not
for public in (nkeys.PREFIX_BYTE_USER, nkeys.PREFIX_BYTE_ACCOUNT, nkeys.PREFIX_BYTE_SERVER):
    assert nkeys.valid_public_prefix_byte(public), public
for secret in (nkeys.PREFIX_BYTE_SEED, nkeys.PREFIX_BYTE_PRIVATE):
    assert nkeys.valid_prefix_byte(secret) and not nkeys.valid_public_prefix_byte(secret), secret
assert set(valid_public) < set(valid_bytes)

print("\nExample 2 completed successfully")

## Example 3: CRC16 checksum

In [ ]:
# Example 3: CRC16 checksum
import nkeys

print("Example 3: CRC16 checksum")

# Test CRC16 checksum function
test_data = [b"Hello, NATS!", b"test data", b"", b"a" * 100]

print("Computing CRC16 checksums:")
for data in test_data:
    checksum = nkeys.crc16_checksum(data)
    data_display = data[:30] if len(data) <= 30 else data[:27] + b"..."
    print(f"  Data: {data_display}")
    print(f"    CRC16: {checksum}")

# Show CRC16 table
print(f"\nCRC16 table length: {len(nkeys.CRC16TAB)}")
print(f"First 5 values: {nkeys.CRC16TAB[:5]}")

print("\nExample 3 completed successfully")

## Example 4: Error types

In [ ]:
# Example 4: Error types
import nkeys

print("Example 4: nkeys error types")

# Show all error types
error_types = [x for x in dir(nkeys) if x.startswith('Err')]
print(f"Available error types ({len(error_types)}): {error_types[:5]}...")

# Show base error class
print(f"\nBase error class: {nkeys.NkeysError}")

# Test error conditions
print("\nTesting error conditions:")
try:
    nkeys.from_seed(b"invalid")
except nkeys.NkeysError as e:
    print(f"  Invalid seed: {type(e).__name__}")
else:
    raise AssertionError("from_seed accepted an invalid seed")

print("\nExample 4 completed successfully")

## Example 5: KeyPair sign and verify

In [ ]:
# Example 5: KeyPair sign and verify
import os

import nkeys

print("Example 5: KeyPair sign and verify")

# Encode 32 random bytes as a user seed and load a KeyPair from it
seed = nkeys.encode_seed(os.urandom(32), nkeys.PREFIX_BYTE_USER)
kp = nkeys.from_seed(bytearray(seed))
print(f"KeyPair class: {type(kp).__name__}")
print(f"Seed prefix: {seed[:2]!r}")
assert seed.startswith(b"SU"), seed[:2]

public_key = kp.public_key
print(f"Public key prefix: {public_key[:1]!r} ({len(public_key)} characters)")
assert public_key.startswith(b"U"), public_key[:1]

# A signature verifies for the signed data and is rejected for anything else
data = b"hello nats"
signature = kp.sign(data)
print(f"Signature length: {len(signature)} bytes")
assert len(signature) == 64
assert kp.verify(data, signature)
try:
    kp.verify(b"tampered", signature)
except nkeys.ErrInvalidSignature:
    print("Tampered data rejected: ErrInvalidSignature")
else:
    raise AssertionError("verify() accepted a signature for different data")

assert hasattr(kp, 'wipe')
kp.wipe()

print("\nExample 5 completed successfully")